# Task 2: Exploratory Data Analysis (EDA)

This section implements only the three required EDA functions.


In [ ]:
# Imports needed for EDA only
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
# Create plot output folder one time.
# This ensures saving plots will not fail if folders are missing.
PLOTS_DIR: Path = Path("assets") / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Fixed seed keeps random sampling reproducible in show_samples.
RANDOM_SEED: int = 42


In [ ]:
def show_samples(df: pd.DataFrame, num_samples: int = 5) -> None:
    """
    Randomly display sample images from the dataset and save the figure.

    The function shows a small visual check of the training data so we can
    quickly confirm that images and labels look correct.
    """

    # Basic input checks help avoid silent errors.
    required_cols = {"folder", "file_name", "label"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")
    if df.empty:
        raise ValueError("Input DataFrame is empty.")
    if num_samples <= 0:
        raise ValueError("num_samples must be greater than 0.")

    # We sample random rows to get representative examples from the dataset.
    n = min(num_samples, len(df))
    sample_df = df.sample(n=n, random_state=RANDOM_SEED).reset_index(drop=True)

    # Build a flexible grid so different num_samples values look clean.
    cols = min(5, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

    # Flatten axes so we can index it easily for both 1-row and multi-row grids.
    axes = np.array(axes).reshape(-1)

    for i, (_, row) in enumerate(sample_df.iterrows()):
        # Build the full image path from folder + file_name columns.
        image_path = Path(row["folder"]) / row["file_name"]

        # Convert to RGB so all images have the same color format.
        # This avoids issues if any image has a different mode.
        image = Image.open(image_path).convert("RGB")

        # Class name is read from the folder name (example: data/Forest -> Forest).
        class_name = Path(row["folder"]).name

        axes[i].imshow(image)
        axes[i].set_title(f"Class: {class_name}", fontsize=9)
        axes[i].axis("off")

    # Hide unused axes if grid has extra spaces.
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()

    # Save the plot to the required location and filename.
    save_path = PLOTS_DIR / "random_samples.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def average_pixel_plot(df: pd.DataFrame) -> None:
    """
    Plot the distribution of per-image average RGB channel values.

    For each image, we compute one mean value for Red, one for Green,
    and one for Blue. Then we visualize these values as histograms.
    """

    # Input validation keeps function behavior predictable.
    required_cols = {"folder", "file_name", "label"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")
    if df.empty:
        raise ValueError("Input DataFrame is empty.")

    # These lists store one average channel value per image.
    red_means: list[float] = []
    green_means: list[float] = []
    blue_means: list[float] = []

    for _, row in df.iterrows():
        # Build full path and load image with PIL.
        image_path = Path(row["folder"]) / row["file_name"]

        # Convert to RGB so channel indexing is consistent for every file.
        image = Image.open(image_path).convert("RGB")

        # Convert image to numpy array for numeric calculations.
        arr = np.array(image, dtype=np.float32)

        # Compute mean pixel value for each channel: R, G, B.
        red_means.append(float(arr[:, :, 0].mean()))
        green_means.append(float(arr[:, :, 1].mean()))
        blue_means.append(float(arr[:, :, 2].mean()))

    # Histogram shows how average channel values are distributed across images.
    plt.figure(figsize=(10, 6))
    plt.hist(red_means, bins=40, alpha=0.5, label="Red", color="red")
    plt.hist(green_means, bins=40, alpha=0.5, label="Green", color="green")
    plt.hist(blue_means, bins=40, alpha=0.5, label="Blue", color="blue")

    plt.title("Distribution of Average Pixel Values")
    plt.xlabel("Average Pixel Value")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save the plot to the required location and filename.
    save_path = PLOTS_DIR / "average_pixel_distribution.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def average_brightness_per_class(df: pd.DataFrame) -> None:
    """
    Create a boxplot of average brightness values per class.

    Brightness is computed as the mean pixel value across all RGB channels
    for each image. The boxplot helps compare class-level brightness spread.
    """

    # Validate input dataframe first.
    required_cols = {"folder", "file_name", "label"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")
    if df.empty:
        raise ValueError("Input DataFrame is empty.")

    # Store brightness values grouped by class name.
    brightness_by_class: dict[str, list[float]] = {}

    for _, row in df.iterrows():
        image_path = Path(row["folder"]) / row["file_name"]

        # RGB conversion keeps brightness calculation consistent.
        image = Image.open(image_path).convert("RGB")
        arr = np.array(image, dtype=np.float32)

        # Brightness = average over all pixels and all RGB channels.
        brightness = float(arr.mean())

        # Use folder name as class name (for readable x-axis labels).
        class_name = Path(row["folder"]).name

        if class_name not in brightness_by_class:
            brightness_by_class[class_name] = []
        brightness_by_class[class_name].append(brightness)

    # Sort class names so plot order is deterministic.
    class_names = sorted(brightness_by_class.keys())
    boxplot_data = [brightness_by_class[c] for c in class_names]

    # Boxplot shows median, quartiles, and spread for each class brightness.
    plt.figure(figsize=(12, 6))
    plt.boxplot(boxplot_data, labels=class_names, showfliers=True)
    plt.title("Average Brightness per Class")
    plt.xlabel("Class")
    plt.ylabel("Average Brightness")

    # Rotate labels so long class names remain easy to read.
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()

    # Save the plot to the required location and filename.
    save_path = PLOTS_DIR / "average_brightness.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# Example usage (works with train_df from Task 1)
show_samples(train_df, num_samples=5)
average_pixel_plot(train_df)
average_brightness_per_class(train_df)
